# 02 Baseline Results

This notebook compares unordered and sequence-aware model performance using saved metrics artifacts from `outputs/metrics/`. It produces tables and plots for your project report.


## Goals

1. Build a clean model comparison table.
2. Compare unordered vs sequence-aware performance on headline metrics.
3. Export report-ready tables/plots.


In [ ]:
import matplotlib
matplotlib.use("Agg")

from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Metrics dir:", METRICS_DIR)
print("Existing JSON files:")
for path in sorted(METRICS_DIR.glob("*_test.json")):
    print(" -", path.name)


In [ ]:
# Load all model metric payloads.
metric_payloads = {}
for path in sorted(METRICS_DIR.glob("*_test.json")):
    with path.open("r", encoding="utf-8") as fp:
        metric_payloads[path.stem] = json.load(fp)

if not metric_payloads:
    raise FileNotFoundError("No *_test.json metric files found. Run experiment scripts first.")

print("Loaded models:", list(metric_payloads.keys()))


In [ ]:
# Flatten high-level metrics into one DataFrame.
rows = []
for model_key, payload in metric_payloads.items():
    overall = payload["metrics"]["overall"]
    row = {
        "model_key": model_key,
        "model_name": payload["model_name"],
        "split": payload["split"],
        "runtime_seconds": payload["metrics"]["runtime_seconds"],
        "num_eval_examples": payload["num_eval_examples"],
        **overall,
    }
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values("HitRate@10", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Classify model families for proposal-oriented comparison.
unordered_models = {"popularity", "cooccurrence", "session_knn_unordered"}

results_df["family"] = np.where(
    results_df["model_name"].isin(unordered_models),
    "unordered",
    "sequence_aware",
)

headline_cols = ["model_name", "family", "HitRate@10", "Recall@20", "MRR@20", "NDCG@20", "runtime_seconds"]
headline_table = results_df[headline_cols].sort_values("HitRate@10", ascending=False)
headline_table


In [ ]:
# Save headline table for the report.
out_table = TABLES_DIR / "baseline_headline_table.csv"
headline_table.to_csv(out_table, index=False)
print("Wrote:", out_table)


In [ ]:
# Plot headline metrics per model.
plot_df = results_df.melt(
    id_vars=["model_name", "family"],
    value_vars=["HitRate@10", "Recall@20", "MRR@20", "NDCG@20"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="model_name", y="score", hue="metric")
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison Across Headline Ranking Metrics")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_metric_comparison.png", dpi=150)
plt.show()


In [ ]:
# Family-level aggregate comparison: unordered vs sequence-aware.
family_summary = (
    results_df.groupby("family")[["HitRate@10", "Recall@20", "MRR@20", "NDCG@20"]]
    .mean()
    .reset_index()
)

family_summary


In [ ]:
# Quantify average gains from sequence-aware models.
unordered_mean = family_summary.loc[family_summary["family"] == "unordered"].iloc[0]
sequence_mean = family_summary.loc[family_summary["family"] == "sequence_aware"].iloc[0]

gain_report = pd.DataFrame(
    {
        "metric": ["HitRate@10", "Recall@20", "MRR@20", "NDCG@20"],
        "unordered_mean": [unordered_mean[m] for m in ["HitRate@10", "Recall@20", "MRR@20", "NDCG@20"]],
        "sequence_aware_mean": [sequence_mean[m] for m in ["HitRate@10", "Recall@20", "MRR@20", "NDCG@20"]],
    }
)
gain_report["absolute_gain"] = gain_report["sequence_aware_mean"] - gain_report["unordered_mean"]
gain_report["relative_gain_pct"] = np.where(
    gain_report["unordered_mean"] > 0,
    100.0 * gain_report["absolute_gain"] / gain_report["unordered_mean"],
    np.nan,
)

gain_report


In [ ]:
# Save gain report for documentation.
out_gain = TABLES_DIR / "sequence_vs_unordered_gain_report.csv"
gain_report.to_csv(out_gain, index=False)
print("Wrote:", out_gain)


In [ ]:
# Build a simple runtime-efficiency view (performance per second).
results_df["HitRate10_per_sec"] = results_df["HitRate@10"] / results_df["runtime_seconds"].clip(lower=1e-9)

efficiency_table = results_df[["model_name", "HitRate@10", "runtime_seconds", "HitRate10_per_sec"]].sort_values(
    "HitRate10_per_sec", ascending=False
)

out_eff = TABLES_DIR / "model_efficiency_table.csv"
efficiency_table.to_csv(out_eff, index=False)
print("Wrote:", out_eff)
efficiency_table


## Interpretation Prompts

1. Which sequence-aware model has the strongest headline metric values?
2. Does the best sequence-aware model beat the strongest unordered baseline?
3. How do gains trade off against runtime?
4. Are results consistent across `HitRate@10`, `MRR@20`, and `NDCG@20`?
